In [1]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.head()

,ID,Name,Age,Gender,Admission Test Score,High School Percentage,City,Admission Status
0,36,Ayesha,24.0,Male,94.0,98.43,Multan,Rejected
1,76,Sajjad,NaN,NaN,90.0,63.04,Quetta,Accepted
2,22,Kamran,18.0,Male,53.0,98.98,Multan,Rejected
3,132,Maaz,20.0,Male,58.0,96.83,Peshawar,Accepted
4,140,Feroze,23.0,Female,51.0,92.36,Rawalpindi,Accepted


In [2]:
df_train.isna().sum()

ID                         0
Name                       5
Age                        4
Gender                     8
Admission Test Score      10
High School Percentage     9
City                       9
Admission Status          10
dtype: int64

In [3]:
df_train.columns

Index(['ID', 'Name', 'Age', 'Gender', 'Admission Test Score',
       'High School Percentage', 'City', 'Admission Status'],
      dtype='object')

In [27]:
#imputing with mean and mode and label encoding
to_impute = [col for col in df_train.columns if col != 'ID' and col != 'Admission Status']

for col in to_impute:
    if df_train[col].dtype == 'float64':
        df_train[col] = df_train[col].fillna(df_train[col].mean())
        df_test[col] = df_test[col].fillna(df_test[col].mean())
    else:
        df_train[col] = df_train[col].fillna(df_train[col].mode()[0])
        df_test[col] = df_test[col].fillna(df_test[col].mode()[0])

df_train["Admission Status"] = df_train["Admission Status"].fillna(df_train["Admission Status"].mode()[0])

to_enc = ['Name', 'Gender', 'City']
from sklearn.preprocessing import TargetEncoder
enc = TargetEncoder()
enc.fit(df_train[to_enc], y)

df_train[to_enc] = enc.transform(df_train[to_enc])
df_test[to_enc] = enc.transform(df_test[to_enc])

df_train.head()

,ID,Name,Age,Gender,Admission Test Score,High School Percentage,City,Admission Status
0,36,1.000000,24.000000,0.615961,94.0,98.43,0.535278,Rejected
1,76,0.000000,19.743363,0.529078,90.0,63.04,0.562549,Accepted
2,22,1.000000,18.000000,0.615961,53.0,98.98,0.535278,Rejected
3,132,0.000000,20.000000,0.615961,58.0,96.83,0.630669,Accepted
4,140,0.386758,23.000000,0.529078,51.0,92.36,0.504340,Accepted


In [28]:
df_train.columns

Index(['ID', 'Name', 'Age', 'Gender', 'Admission Test Score',
       'High School Percentage', 'City', 'Admission Status'],
      dtype='object')

In [29]:
num_features = ['Age', 'Admission Test Score', 'High School Percentage']

In [30]:
features = [col for col in df_train.columns if col != 'ID' and col != 'Admission Status']
target = 'Admission Status'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

X.head()

,Name,Age,Gender,Admission Test Score,High School Percentage,City
0,1.000000,24.000000,0.615961,94.0,98.43,0.535278
1,0.000000,19.743363,0.529078,90.0,63.04,0.562549
2,1.000000,18.000000,0.615961,53.0,98.98,0.535278
3,0.000000,20.000000,0.615961,58.0,96.83,0.630669
4,0.386758,23.000000,0.529078,51.0,92.36,0.504340


In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

model = LogisticRegression(random_state=42)

cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()

0.8217391304347826

In [41]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [42]:
df_preds = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.ID,
    "answer": y_preds
})

df_preds.tail()

,subtaskID,datapointID,answer
35,1,127,Rejected
36,1,26,Accepted
37,1,135,Accepted
38,1,78,Rejected
39,1,148,Accepted


In [43]:
df_preds.to_csv("submission.csv", index=False)